# Explorando o Data Lakehouse

Este notebook é o laboratório exploratório da aula. Tudo já está conectado — você não precisa configurar host, porta, usuário ou senha de nada.

O pipeline bronze -> silver -> gold já roda sozinho, uma vez, assim que o ambiente sobe (serviço `pipeline-init`, ver `trino/init/construir_pipeline.py`) — então as tabelas abaixo já existem, sem você precisar rodar nada antes. Se quiser reconstruir depois de inserir dado novo no Postgres, veja a seção 6 no fim deste notebook.

Enquanto ele roda (na subida do ambiente, ou se você rodar de novo na mão), dá pra acompanhar visualmente em duas telas:
- **MinIO Console** (http://localhost:9001) — vendo os arquivos Parquet aparecerem em `lakehouse/bronze`, `/silver`, `/gold`.
- **Trino UI** (http://localhost:8082) — vendo as queries `CREATE TABLE ... AS SELECT` rodando.


In [ ]:
import lakehouse_kit as lh
import matplotlib.pyplot as plt

## 1. O armazenamento bruto (MinIO)

Por baixo de tudo, o lakehouse é só um bucket S3 com pastas. Vamos listar o que existe fisicamente na camada bronze:

In [ ]:
lh.list_layer("bronze")

## 2. Consultando com SQL via Trino

O Trino enxerga esses arquivos como tabelas de verdade, organizadas em schemas (`bronze`, `silver`, `gold`) dentro do catálogo `lakehouse`.

In [ ]:
lh.query("SHOW SCHEMAS FROM lakehouse")

In [ ]:
# Camada bronze: dado cru, exatamente como veio da fonte
lh.query("SELECT * FROM lakehouse.bronze.customers LIMIT 10")


## 3. Camada Silver — dado limpo e unificado

Pedidos + itens + produtos + clientes já vêm juntados num único modelo, pronto para análise:

In [ ]:
lh.query("SELECT * FROM lakehouse.silver.sales LIMIT 10")

## 4. Camada Gold — agregados prontos para consumo

In [ ]:
gold_por_dia = lh.query("SELECT * FROM lakehouse.gold.sales_by_day ORDER BY order_date")
gold_por_dia

In [ ]:
gold_por_dia.plot(x="order_date", y="receita", kind="line", marker="o", figsize=(10, 4), title="Receita por dia")
plt.tight_layout()
plt.show()

In [ ]:
gold_por_categoria = lh.query("SELECT * FROM lakehouse.gold.sales_by_category ORDER BY receita DESC")
gold_por_categoria.plot(x="product_category", y="receita", kind="bar", figsize=(8, 4), title="Receita por categoria", legend=False)
plt.tight_layout()
plt.show()

## 5. Construindo sua própria camada (write_table / read_table)

Tudo que você viu até aqui (bronze, silver, gold) foi construído pelo `pipeline-init` chamando exatamente as mesmas funções que você tem disponíveis agora — não tem nada de especial acontecendo por trás.

- `lh.query(...)` (usado acima) sempre passa pelo **Trino** — só enxerga o que já foi cadastrado no catálogo.
- `lh.read_table(layer, tabela)` lê o Parquet direto do **MinIO** com pandas, sem passar pelo Trino — útil pra inspecionar um arquivo antes (ou mesmo sem nunca) cadastrar.
- `lh.write_table(df, layer, tabela)` grava um DataFrame como Parquet no MinIO **e** cadastra a tabela no catálogo do Trino na mesma chamada — depois disso, `lh.query(...)` já enxerga. Os tipos das colunas são adivinhados a partir do DataFrame; pra forçar um tipo específico, passe `colunas_sql="coluna TIPO, ..."` (mesmo formato usado em `trino/init/construir_pipeline.py`).
- `lh.drop_table(layer, tabela)` desfaz: remove a tabela do catálogo e os arquivos do MinIO.

É assim que se constrói qualquer camada da arquitetura medalhão "na mão": lê de uma origem qualquer (Postgres, outra tabela via `lh.query`, um CSV, uma API...), transforma com pandas à vontade, e publica com `write_table`. Exemplo — um "top 3 clientes por receita" que não existe no pipeline oficial:

In [ ]:
# Lê a silver via SQL, agrega com pandas, publica como uma tabela gold nova
top_clientes = lh.query("SELECT customer_name, SUM(item_total) AS receita FROM lakehouse.silver.sales GROUP BY customer_name")
top_clientes = top_clientes.sort_values("receita", ascending=False).head(3).reset_index(drop=True)

lh.write_table(top_clientes, "gold", "top_clientes")
lh.query("SELECT * FROM lakehouse.gold.top_clientes")

In [ ]:
# lh.drop_table("gold", "top_clientes")  # descomente pra desfazer (tira do catálogo E apaga o Parquet)

## 6. Para explorar em aula

- Insira um novo pedido direto no Postgres (`lh.postgres()` + um `INSERT`) e rode o pipeline de novo pra atualizar o lakehouse — de dentro deste notebook, numa célula nova: `!python /home/jovyan/trino-init/construir_pipeline.py`. Depois, rode de novo as células deste notebook e veja o `gold.sales_by_day` mudar.
- Abra o MinIO Console e compare o tamanho/quantidade de arquivos entre bronze e gold — por que gold tem menos dado?
- Escreva uma nova query de agregação (ex: receita por cliente) direto em `lh.query(...)`.
- Escreva sua própria camada com `lh.write_table(...)` (seção 5) — ex: uma tabela `bronze` a partir de um CSV que você mesmo suba pro Jupyter, ou uma camada `silver`/`gold` alternativa a partir de outra pergunta de negócio.
- Derrube tudo com `docker compose down -v` e suba de novo — o ambiente inteiro volta ao estado zero (e o pipeline roda sozinho de novo).
